In [17]:
# ============================================================================
# Watershed Eligibility Analysis – Current Green Cover Only
# For use in Jupyter Notebook with geemap / Earth Engine
# ============================================================================

import geemap
import ee

# Initialize Earth Engine (authenticate if needed)
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

# Create interactive map
Map = geemap.Map()
Map.add_basemap('HYBRID')

# 1. Define your area of interest (replace with your own polygon asset or geometry)
roi = ee.FeatureCollection("projects/ee-clivedcosta/assets/Watershed")  # example
# Alternatively, draw on map and get geometry:
# roi = Map.draw_last_feature.geometry()
Map.centerObject(roi, 12)
Map.addLayer(roi, {'color': 'yellow'}, 'Watershed Boundary')

# 2. Parameters (adjust as needed)
NDVI_THRESHOLD = 0.45          # NDVI > 0.3 = dense green cover (exclude)
CANOPY_THRESHOLD = 2          # canopy height >=2 m = trees (exclude)
CLOUD_COVER_MAX = 20          # maximum cloud cover % per Sentinel-2 scene
START_DATE = '2025-01-01'
END_DATE = '2025-12-01'

# ============================================================================
# 3. Sentinel-2 NDVI – current green cover
# Source: COPERNICUS/S2_SR (Level-2A surface reflectance, 10 m resolution)
# ============================================================================
s2 = ee.ImageCollection('COPERNICUS/S2_SR') \
    .filterBounds(roi) \
    .filterDate(START_DATE, END_DATE) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_COVER_MAX))

def add_ndvi(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return image.addBands(ndvi)

s2_with_ndvi = s2.map(add_ndvi)
# Median composite to reduce clouds and shadows
s2_composite = s2_with_ndvi.median().clip(roi)
ndvi = s2_composite.select('NDVI')

# Pixels with dense green cover (excluded)
dense_green = ndvi.gte(NDVI_THRESHOLD)
Map.addLayer(dense_green.selfMask(), {'palette': ['green']}, 'Dense green (excluded)', False)

# ============================================================================
# 4. Global Canopy Height (Meta / ETH Zurich)
# Source: projects/meta-forest-monitoring-okw37/assets/CanopyHeight
# 10 m resolution, derived from GEDI + Sentinel-2
# ============================================================================
canopy = ee.ImageCollection('projects/meta-forest-monitoring-okw37/assets/CanopyHeight') \
    .mosaic() \
    .clip(roi) \
    .select('cover_code')   # band with canopy height in meters

tall_trees = canopy.gte(CANOPY_THRESHOLD)
Map.addLayer(tall_trees.selfMask(), {'palette': ['darkgreen']}, 'Tall trees (>=2m, excluded)', False)

# ============================================================================
# 5. Combine exclusions and compute eligible areas
# Eligible = NOT (dense green OR tall trees)
# ============================================================================
excluded = dense_green.Or(tall_trees)
eligible = excluded.Not().selfMask()   # selfMask() makes 1-values opaque

Map.addLayer(eligible, {'palette': ['00FF00']}, 'Eligible Areas (raster)', True)

# Convert eligible raster to vector polygons (optional)
eligible_vectors = eligible.reduceToVectors(
    geometry=roi.geometry(),
    scale=10,
    geometryType='polygon',
    eightConnected=False,
    bestEffort=True,
    maxPixels=1e13
)
Map.addLayer(eligible_vectors, {'color': '00FF00', 'fillColor': '00FF0044'}, 'Eligible Polygons')

# ============================================================================
# 6. Area calculations (hectares)
# ============================================================================
pixel_area = ee.Image.pixelArea()   # m² per pixel

total_area_ha = roi.geometry().area().divide(10000)

dense_green_area_ha = dense_green.multiply(pixel_area) \
    .reduceRegion(reducer=ee.Reducer.sum(), geometry=roi, scale=10, maxPixels=1e13) \
    .getNumber('NDVI').divide(10000)

tall_trees_area_ha = tall_trees.multiply(pixel_area) \
    .reduceRegion(reducer=ee.Reducer.sum(), geometry=roi, scale=10, maxPixels=1e13) \
    .getNumber('cover_code').divide(10000)

eligible_area_ha = eligible.multiply(pixel_area) \
    .reduceRegion(reducer=ee.Reducer.sum(), geometry=roi, scale=10, maxPixels=1e13) \
    .getNumber('NDVI').divide(10000)

# Fetch and print results
total_ha = total_area_ha.getInfo()
dense_ha = dense_green_area_ha.getInfo()
tall_ha = tall_trees_area_ha.getInfo()
eligible_ha = eligible_area_ha.getInfo()

print("=== WATERSHED ELIGIBILITY REPORT (Current Green Cover Only) ===")
print(f"Total area: {total_ha:.2f} ha")
print(f"Excluded - Dense green (NDVI > {NDVI_THRESHOLD}): {dense_ha:.2f} ha")
print(f"Excluded - Tall trees (canopy ≥ {CANOPY_THRESHOLD} m): {tall_ha:.2f} ha")
print(f"Eligible area: {eligible_ha:.2f} ha")
print(f"Percentage eligible: {eligible_ha/total_ha*100:.1f}%")

# ============================================================================
# 7. Export eligible polygons (Shapefile / KML / GeoJSON)
# ============================================================================
export_task = ee.batch.Export.table.toDrive(
    collection=eligible_vectors,
    description='Watershed_Eligible_CurrentGreen',
    folder='Watershed_Analysis',
    fileFormat='SHP'   # Change to 'KML' or 'GeoJSON' if desired
)
export_task.start()
print("Export started. Check Tasks tab in Earth Engine.")

# Display map
Map.addLayerControl()
Map

=== WATERSHED ELIGIBILITY REPORT (Current Green Cover Only) ===
Total area: 67.48 ha
Excluded - Dense green (NDVI > 0.45): 0.07 ha
Excluded - Tall trees (canopy ≥ 2 m): 17.19 ha
Eligible area: 49.97 ha
Percentage eligible: 74.0%
Export started. Check Tasks tab in Earth Engine.


Map(center=[12.863556182712198, 77.87145656245913], controls=(WidgetControl(options=['position', 'transparent_…

In [18]:
import geemap
import ee
import pandas as pd
import numpy as np
from scipy import stats
import plotly.graph_objects as go

# Initialize Earth Engine
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

Map = geemap.Map()
Map.add_basemap('HYBRID')

# Study area
roi = ee.FeatureCollection("projects/ee-clivedcosta/assets/Odishamerged")
Map.centerObject(roi, 10)
Map.addLayer(roi, {'color': 'yellow'}, 'Watershed Boundary', False)

# Parameters
START_DATE = '2024-01-01'
END_DATE = '2025-12-31'
SCALE = 1000

def create_monthly_composites(image_collection, start_date, end_date, band_name):
    start = ee.Date(start_date)
    end = ee.Date(end_date)
    months = end.difference(start, 'month').round()
    def make_monthly_image(offset):
        month_start = start.advance(offset, 'month')
        month_end = month_start.advance(1, 'month')
        monthly_mean = image_collection \
            .filterDate(month_start, month_end) \
            .mean() \
            .rename(band_name) \
            .set('system:time_start', month_start.millis())
        return monthly_mean
    offsets = ee.List.sequence(0, ee.Number(months).subtract(1))
    return ee.ImageCollection(offsets.map(make_monthly_image))

# Load GLDAS
print("Loading GLDAS groundwater data...")
try:
    gws_v22 = ee.ImageCollection("NASA/GLDAS/V022/CLSM/G025/DA1D") \
        .select('GWS_tavg') \
        .filterDate(START_DATE, END_DATE)
    count = gws_v22.size().getInfo()
    if count > 0:
        gws = gws_v22
        print(f"  Using GLDAS-2.2 (Nooh) – {count} daily images")
    else:
        raise Exception("No data")
except:
    gws = ee.ImageCollection("NASA/GLDAS/V021/CLSM/G025/DA1D") \
        .select('GWS_tavg') \
        .filterDate(START_DATE, END_DATE)
    count = gws.size().getInfo()
    print(f"  Using GLDAS-2.1 – {count} daily images")

monthly_gws = create_monthly_composites(gws, START_DATE, END_DATE, 'groundwater_storage')

def extract_series(image_collection, band_name, region, scale):
    def get_mean(img):
        mean_val = img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=region,
            scale=scale,
            maxPixels=1e13,
            bestEffort=True
        )
        date_str = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd')
        return ee.Feature(None, {'date': date_str, 'value': mean_val.get(band_name)})
    return image_collection.map(get_mean)

gws_fc = extract_series(monthly_gws, 'groundwater_storage', roi, SCALE)

def fc_to_dataframe(fc):
    if fc.size().getInfo() == 0:
        return pd.DataFrame()
    data = fc.getInfo()
    records = []
    for feat in data['features']:
        props = feat['properties']
        if props.get('value') is not None:
            records.append({'date': pd.to_datetime(props['date']), 'groundwater_storage_mm': props['value']})
    df = pd.DataFrame(records)
    if not df.empty:
        df = df.sort_values('date')
    return df

gws_df = fc_to_dataframe(gws_fc)

# Time series plot
if not gws_df.empty:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=gws_df['date'],
        y=gws_df['groundwater_storage_mm'],
        mode='lines+markers',
        name='Groundwater Storage (GLDAS)',
        line=dict(color='blue', width=2),
        marker=dict(size=6)
    ))
    fig.update_layout(
        title='Groundwater Storage Time Series (2024-2025)',
        xaxis_title='Date',
        yaxis_title='Storage (mm)',
        hovermode='x unified',
        template='plotly_white'
    )
    fig.show()
else:
    print("No groundwater data available.")

# ----- FIXED LEGEND MAP LAYER (0-1000 mm) -----
latest_gws = monthly_gws.sort('system:time_start', False).first().clip(roi)

min_fixed = 0
max_fixed = 1000    # adjust if your values are higher, e.g. 1500

gws_vis = {
    'min': min_fixed,
    'max': max_fixed,
    'palette': ['red', 'orange', 'yellow', 'lightgreen', 'lightblue', 'blue']
}
Map.addLayer(latest_gws, gws_vis, 'Latest Groundwater Storage (mm)')
Map.add_colorbar(
    vis_params=gws_vis,
    label='Groundwater Storage (mm)',
    layer_name='Latest Groundwater Storage (mm)',
    orientation='vertical'
)
print(f"Legend uses fixed range: {min_fixed} - {max_fixed} mm (red = low/dry, blue = high/wet)")

# Statistical report
print("\n" + "="*60)
print("GROUNDWATER MONITORING REPORT")
print(f"Period: {START_DATE} to {END_DATE}")
print("="*60)

if not gws_df.empty:
    print(f"\nGLDAS Groundwater Storage (mm):")
    print(f"  Mean: {gws_df['groundwater_storage_mm'].mean():.2f}")
    print(f"  Min:  {gws_df['groundwater_storage_mm'].min():.2f}")
    print(f"  Max:  {gws_df['groundwater_storage_mm'].max():.2f}")
    print(f"  Std:  {gws_df['groundwater_storage_mm'].std():.2f}")
    
    x = np.arange(len(gws_df))
    y = gws_df['groundwater_storage_mm'].values
    slope, _, r_value, p_value, _ = stats.linregress(x, y)
    trend_mm_per_year = slope * 12
    print(f"\nLinear Trend: {trend_mm_per_year:.2f} mm/year")
    print(f"  R-squared: {r_value**2:.3f}")
    print(f"  p-value: {p_value:.4f}")
    if p_value < 0.05:
        print("  ✓ Statistically significant trend (p < 0.05)")
    else:
        print("  ✗ Trend not statistically significant")
    if trend_mm_per_year > 5:
        print("  → Groundwater increasing noticeably.")
    elif trend_mm_per_year < -5:
        print("  → Groundwater decreasing noticeably.")
    else:
        print("  → Groundwater relatively stable.")

print("\n" + "="*60)
print("DATA SOURCE: NASA GLDAS (GWS_tavg – groundwater storage)")
print("Fixed legend range: 0-1000 mm (adjustable in code)")
print("="*60)

Map

Loading GLDAS groundwater data...
  Using GLDAS-2.2 (Nooh) – 730 daily images


Legend uses fixed range: 0 - 1000 mm (red = low/dry, blue = high/wet)

GROUNDWATER MONITORING REPORT
Period: 2024-01-01 to 2025-12-31

GLDAS Groundwater Storage (mm):
  Mean: 770.10
  Min:  575.82
  Max:  983.06
  Std:  147.14

Linear Trend: 106.30 mm/year
  R-squared: 0.181
  p-value: 0.0381
  ✓ Statistically significant trend (p < 0.05)
  → Groundwater increasing noticeably.

DATA SOURCE: NASA GLDAS (GWS_tavg – groundwater storage)
Fixed legend range: 0-1000 mm (adjustable in code)


Map(center=[22.047020860523727, 86.31533379126039], controls=(WidgetControl(options=['position', 'transparent_…

In [19]:
import geemap
import ee
import pandas as pd
import plotly.graph_objects as go

try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

Map = geemap.Map()
Map.add_basemap('HYBRID')

roi = ee.FeatureCollection("projects/ee-clivedcosta/assets/Watershed")
Map.centerObject(roi, 10)
Map.addLayer(roi, {'color': 'yellow'}, 'Watershed Boundary', False)

# Load GPM IMERG V07 (precipitation rate, mm/hr, half-hourly)
rainfall_imerg = ee.ImageCollection("NASA/GPM_L3/IMERG_V07") \
    .select('precipitation') \
    .filterDate('2025-01-01', '2025-12-31') \
    .filterBounds(roi)

def daily_total_from_halfhourly(collection, start_date, end_date):
    start = ee.Date(start_date)
    end = ee.Date(end_date)
    days = end.difference(start, 'day').round()
    def make_daily_image(offset):
        day_start = start.advance(offset, 'day')
        day_end = day_start.advance(1, 'day')
        daily_sum = collection.filterDate(day_start, day_end) \
            .reduce(ee.Reducer.sum()) \
            .multiply(0.5) \
            .rename('daily_precipitation') \
            .set('system:time_start', day_start.millis())
        return daily_sum
    days_list = ee.List.sequence(0, ee.Number(days).subtract(1))
    return ee.ImageCollection(days_list.map(make_daily_image))

daily_rainfall = daily_total_from_halfhourly(rainfall_imerg, '2025-01-01', '2025-12-31')

def extract_series(image_collection, band_name, region, scale):
    def get_mean(img):
        mean_val = img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=region,
            scale=scale,
            maxPixels=1e13,
            bestEffort=True
        )
        date_str = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd')
        return ee.Feature(None, {'date': date_str, 'value': mean_val.get(band_name)})
    return image_collection.map(get_mean)

rainfall_fc = extract_series(daily_rainfall, 'daily_precipitation', roi, 1000)

def fc_to_dataframe(fc):
    if fc.size().getInfo() == 0:
        return pd.DataFrame()
    data = fc.getInfo()
    records = []
    for feat in data['features']:
        props = feat['properties']
        if props.get('value') is not None:
            records.append({'date': pd.to_datetime(props['date']), 'rainfall_mm': props['value']})
    df = pd.DataFrame(records)
    return df.sort_values('date')

rain_df = fc_to_dataframe(rainfall_fc)

# Time series plot
if not rain_df.empty:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=rain_df['date'], y=rain_df['rainfall_mm'],
        mode='lines', name='Daily Rainfall (GPM IMERG)',
        line=dict(color='blue', width=1.5), fill='tozeroy'
    ))
    fig.update_layout(title='Daily Rainfall Time Series (2025)', xaxis_title='Date',
                      yaxis_title='Precipitation (mm/day)', hovermode='x unified', template='plotly_white')
    fig.show()

# Annual rainfall map with corrected palette (red = low, blue = high)
annual_rainfall = daily_rainfall.sum().clip(roi)
min_rain, max_rain = 0, 2000   # adjust max if needed
rain_vis = {
    'min': min_rain,
    'max': max_rain,
    'palette': ['red', 'orange', 'yellow', 'green', 'cyan', 'blue']
}
Map.addLayer(annual_rainfall, rain_vis, 'Annual Rainfall 2025 (mm)')
Map.add_colorbar(vis_params=rain_vis, label='Annual Rainfall (mm)',
                 layer_name='Annual Rainfall 2025 (mm)', orientation='vertical')

# Statistics
if not rain_df.empty:
    total = rain_df['rainfall_mm'].sum()
    mean_daily = rain_df['rainfall_mm'].mean()
    max_daily = rain_df['rainfall_mm'].max()
    max_date = rain_df.loc[rain_df['rainfall_mm'].idxmax(), 'date']
    print("\n" + "="*60)
    print("RAINFALL SUMMARY REPORT (2025)")
    print("="*60)
    print(f"Total Annual Rainfall: {total:.1f} mm")
    print(f"Mean Daily Rainfall: {mean_daily:.2f} mm/day")
    print(f"Maximum Daily Rainfall: {max_daily:.1f} mm on {max_date.strftime('%Y-%m-%d')}")
    print("="*60)

Map


RAINFALL SUMMARY REPORT (2025)
Total Annual Rainfall: 1137.8 mm
Mean Daily Rainfall: 3.13 mm/day
Maximum Daily Rainfall: 69.4 mm on 2025-09-18


Map(center=[12.863556182712198, 77.87145656245913], controls=(WidgetControl(options=['position', 'transparent_…

In [20]:
import geemap
import ee
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy import stats

# Initialize Earth Engine
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

Map = geemap.Map()
Map.add_basemap('HYBRID')

# Study area
roi = ee.FeatureCollection("projects/ee-clivedcosta/assets/Watershed")
Map.centerObject(roi, 10)
Map.addLayer(roi, {'color': 'yellow'}, 'Watershed Boundary', False)

# Parameters
START_DATE = '2024-01-01'
END_DATE = '2025-12-31'
CLOUD_FILTER = 60   # max cloud cover % for scene selection
SCALE = 10

# ------------------------------------------------------------------
# 1. Load Sentinel-2 harmonized collection
# ------------------------------------------------------------------
sentinel = ee.ImageCollection("COPERNICUS/S2_HARMONIZED") \
    .filterBounds(roi) \
    .filterDate(START_DATE, END_DATE) \
    .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER))

# ------------------------------------------------------------------
# 2. Cloud masking using QA60 band (bits 10 and 11)
# ------------------------------------------------------------------
def mask_clouds(image):
    """Mask clouds and cirrus using QA60 band (bit10 = cloud, bit11 = cirrus)."""
    qa60 = image.select('QA60')
    # Bits 10 and 11 are cloud and cirrus; we clear them.
    cloud_bit_mask = (1 << 10) | (1 << 11)
    mask = qa60.bitwiseAnd(cloud_bit_mask).eq(0)
    return image.updateMask(mask)

sentinel_clean = sentinel.map(mask_clouds)

# ------------------------------------------------------------------
# 3. Compute NDMI = (B8 - B11) / (B8 + B11)
# ------------------------------------------------------------------
def add_ndmi(image):
    ndmi = image.normalizedDifference(['B8', 'B11']).rename('NDMI')
    return ndmi.copyProperties(image, ['system:time_start'])

ndmi_collection = sentinel_clean.map(add_ndmi).select('NDMI')

# ------------------------------------------------------------------
# 4. Monthly median composites
# ------------------------------------------------------------------
def create_monthly_composites(collection, start_date, end_date):
    start = ee.Date(start_date)
    end = ee.Date(end_date)
    months = end.difference(start, 'month').round()
    def make_monthly_image(offset):
        month_start = start.advance(offset, 'month')
        month_end = month_start.advance(1, 'month')
        monthly = collection.filterDate(month_start, month_end) \
                    .median() \
                    .rename('NDMI') \
                    .set('system:time_start', month_start.millis())
        return monthly
    offsets = ee.List.sequence(0, ee.Number(months).subtract(1))
    return ee.ImageCollection(offsets.map(make_monthly_image))

monthly_ndmi = create_monthly_composites(ndmi_collection, START_DATE, END_DATE)
print(f"Monthly composites: {monthly_ndmi.size().getInfo()}")

# ------------------------------------------------------------------
# 5. Extract mean NDMI over ROI per month (safe dictionary access)
# ------------------------------------------------------------------
def extract_series_safe(collection, band_name, region, scale):
    def get_mean(img):
        mean_dict = img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=region,
            scale=scale,
            maxPixels=1e13,
            bestEffort=True
        )
        # Provide default -999 if band missing (no valid pixels)
        value = ee.Dictionary(mean_dict).get(band_name, -999)
        date_str = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd')
        return ee.Feature(None, {'date': date_str, 'value': value})
    return collection.map(get_mean)

ndmi_fc = extract_series_safe(monthly_ndmi, 'NDMI', roi, SCALE)

def fc_to_dataframe_safe(fc):
    if fc.size().getInfo() == 0:
        return pd.DataFrame()
    data = fc.getInfo()
    records = []
    for feat in data['features']:
        props = feat['properties']
        val = props.get('value')
        if val is not None and val != -999:
            records.append({'date': pd.to_datetime(props['date']), 'ndmi': val})
    df = pd.DataFrame(records)
    if not df.empty:
        df = df.sort_values('date')
    return df

df_ndmi = fc_to_dataframe_safe(ndmi_fc)
print(f"Valid months with NDMI data: {len(df_ndmi)}")

# ------------------------------------------------------------------
# 6. Time series plot
# ------------------------------------------------------------------
if not df_ndmi.empty:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_ndmi['date'], y=df_ndmi['ndmi'],
        mode='lines+markers', name='NDMI',
        line=dict(color='blue', width=2), marker=dict(size=6)
    ))
    fig.update_layout(title='NDMI Time Series (2024-2025)', xaxis_title='Date',
                      yaxis_title='NDMI', hovermode='x unified', template='plotly_white')
    fig.show()
else:
    print("No valid NDMI data after cloud masking.")

# ------------------------------------------------------------------
# 7. Trend analysis
# ------------------------------------------------------------------
if len(df_ndmi) > 5:
    x = np.arange(len(df_ndmi))
    y = df_ndmi['ndmi'].values
    from scipy.stats import theilslopes, kendalltau
    slope, _, _, _ = theilslopes(y, x)
    tau, p_value = kendalltau(x, y)
    print("\n" + "="*60)
    print("NDMI TREND ANALYSIS")
    print("="*60)
    print(f"Theil-Sen slope per month: {slope:.5f}")
    print(f"Slope per year: {slope*12:.4f}")
    print(f"Mann-Kendall p-value: {p_value:.4f}")
    if p_value < 0.05:
        print("✓ Statistically significant trend (p < 0.05)")
    else:
        print("✗ Trend not significant")

# ------------------------------------------------------------------
# 8. Map: Latest NDMI composite (red = dry, blue = wet)
# ------------------------------------------------------------------
latest_ndmi = monthly_ndmi.sort('system:time_start', False).first().clip(roi)
ndmi_vis = {'min': -1, 'max': 1, 'palette': ['red', 'orange', 'yellow', 'lightgreen', 'lightblue', 'blue']}
Map.addLayer(latest_ndmi, ndmi_vis, 'Latest NDMI')
Map.add_colorbar(vis_params=ndmi_vis, label='NDMI', layer_name='Latest NDMI', orientation='vertical')

# ------------------------------------------------------------------
# 9. Summary statistics
# ------------------------------------------------------------------
if not df_ndmi.empty:
    print("\n" + "="*60)
    print("NDMI SUMMARY")
    print("="*60)
    print(f"Mean NDMI: {df_ndmi['ndmi'].mean():.3f}")
    print(f"Min NDMI:  {df_ndmi['ndmi'].min():.3f}")
    print(f"Max NDMI:  {df_ndmi['ndmi'].max():.3f}")
    print(f"Std:       {df_ndmi['ndmi'].std():.3f}")
    print("="*60)

Map

Monthly composites: 24
Valid months with NDMI data: 22



NDMI TREND ANALYSIS
Theil-Sen slope per month: 0.01265
Slope per year: 0.1518
Mann-Kendall p-value: 0.0039
✓ Statistically significant trend (p < 0.05)



NDMI SUMMARY
Mean NDMI: 0.007
Min NDMI:  -0.184
Max NDMI:  0.194
Std:       0.124


Map(center=[12.863556182712198, 77.87145656245913], controls=(WidgetControl(options=['position', 'transparent_…

In [21]:
"""
============================================================
  SOIL MOISTURE ANALYSIS — Odisha Watershed
  Datasets: SMAP L4 (SPL4SMGP/007) + SMAP L3 Enhanced (SPL3SMP_E/005)
============================================================
"""

# ─────────────────────────────────────────────
# 0.  IMPORTS
# ─────────────────────────────────────────────
import ee
import geemap
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# 1.  EARTH ENGINE INIT
# ─────────────────────────────────────────────
try:
    ee.Initialize()
    print("✅ Earth Engine initialised")
except Exception:
    ee.Authenticate()
    ee.Initialize()
    print("✅ Earth Engine authenticated & initialised")

# ─────────────────────────────────────────────
# 2.  STUDY AREA & PARAMETERS
# ─────────────────────────────────────────────
roi = ee.FeatureCollection("projects/ee-clivedcosta/assets/Watershed")

START_DATE = '2024-01-01'
END_DATE   = '2024-12-31'
SCALE_L4   = 11000   # ~9 km native → 11 km safe
SCALE_L3   = 36000   # 36 km native

# ─────────────────────────────────────────────
# 3.  LOAD DATASETS
# ─────────────────────────────────────────────
print("Loading SMAP L4 (SPL4SMGP v007)…")
smap_l4_raw = (
    ee.ImageCollection("NASA/SMAP/SPL4SMGP/007")
    .filterDate(START_DATE, END_DATE)
    .select(['sm_surface', 'sm_rootzone'])
)
print(f"   L4 images found: {smap_l4_raw.size().getInfo()}")

print("Loading SMAP L3 Enhanced (SPL3SMP_E v005)…")
smap_l3_raw = (
    ee.ImageCollection("NASA/SMAP/SPL3SMP_E/005")
    .filterDate(START_DATE, END_DATE)
    .select('soil_moisture_am')
)
print(f"   L3 images found: {smap_l3_raw.size().getInfo()}")

# ─────────────────────────────────────────────
# 4.  MONTHLY COMPOSITES
# ─────────────────────────────────────────────
def monthly_composites(col, start, end, bands):
    """Monthly mean composites. Empty months return a -9999 fill image."""
    start_ee = ee.Date(start)
    end_ee   = ee.Date(end)
    n_months = end_ee.difference(start_ee, 'month').round()
    offsets  = ee.List.sequence(0, ee.Number(n_months).subtract(1))

    n_bands    = len(bands) if isinstance(bands, list) else 1
    fill_image = ee.Image.constant([-9999] * n_bands).rename(bands)

    def make_month(offset):
        m_start  = start_ee.advance(offset, 'month')
        m_end    = m_start.advance(1, 'month')
        filtered = col.filterDate(m_start, m_end)
        has_data = filtered.size().gt(0)
        monthly  = filtered.mean().rename(bands)
        safe     = ee.Image(ee.Algorithms.If(has_data, monthly, fill_image))
        return safe.set('system:time_start', m_start.millis())

    return ee.ImageCollection(offsets.map(make_month))

monthly_l4_surface  = monthly_composites(
    smap_l4_raw.select('sm_surface'),  START_DATE, END_DATE, ['sm_surface'])
monthly_l4_rootzone = monthly_composites(
    smap_l4_raw.select('sm_rootzone'), START_DATE, END_DATE, ['sm_rootzone'])
monthly_l3          = monthly_composites(
    smap_l3_raw, START_DATE, END_DATE, ['sm_l3_am'])

# ─────────────────────────────────────────────
# 5.  EXTRACT TIME-SERIES TO DATAFRAME
# ─────────────────────────────────────────────
def extract_series(col, band, region, scale, col_name):
    def get_mean(img):
        val = img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=region,
            scale=scale,
            maxPixels=1e13,
            bestEffort=True
        )
        return ee.Feature(None, {
            'date':  ee.Date(img.get('system:time_start')).format('YYYY-MM-dd'),
            'value': val.get(band, None)
        })
    fc = col.map(get_mean)
    records = [
        {'date': f['properties']['date'], col_name: f['properties']['value']}
        for f in fc.getInfo()['features']
        if f['properties'].get('value') is not None
        and f['properties']['value'] != -9999
    ]
    if not records:
        print(f"  ⚠️  No valid data for '{col_name}'")
        return pd.DataFrame(columns=['date', col_name])
    df = pd.DataFrame(records)
    df['date'] = pd.to_datetime(df['date'])
    return df.sort_values('date').reset_index(drop=True)

print("\nExtracting time series (this may take ~1-2 min)…")
df_surface  = extract_series(monthly_l4_surface,  'sm_surface',  roi, SCALE_L4, 'sm_surface_m3m3')
df_rootzone = extract_series(monthly_l4_rootzone, 'sm_rootzone', roi, SCALE_L4, 'sm_rootzone_m3m3')
df_l3       = extract_series(monthly_l3,          'sm_l3_am',    roi, SCALE_L3, 'sm_l3_m3m3')

# Merge into one dataframe
df = df_surface.merge(df_rootzone, on='date', how='outer') \
               .merge(df_l3,       on='date', how='outer') \
               .sort_values('date').reset_index(drop=True)
df['month_label'] = df['date'].dt.strftime('%b %Y')

print(f"✅ Extracted {len(df)} monthly records")

# ─────────────────────────────────────────────
# 6.  CHART — Monthly Soil Moisture Comparison
# ─────────────────────────────────────────────
COLORS = {
    'surface':  '#2196F3',   # blue
    'rootzone': '#4CAF50',   # green
    'l3':       '#FF9800',   # amber
    'bg':       '#F8F9FA',
    'grid':     '#E0E0E0',
}

fig = go.Figure()

for col, label, color in [
    ('sm_surface_m3m3',  'Surface (0–5 cm)',     COLORS['surface']),
    ('sm_rootzone_m3m3', 'Root-Zone (0–100 cm)', COLORS['rootzone']),
    ('sm_l3_m3m3',       'L3 Reference (36 km)', COLORS['l3']),
]:
    tmp = df.dropna(subset=[col])
    fig.add_trace(go.Bar(
        x=tmp['month_label'],
        y=tmp[col],
        name=label,
        marker_color=color,
        opacity=0.87,
        hovertemplate='%{x}<br>' + label + ': %{y:.4f} m³/m³<extra></extra>'
    ))

fig.update_layout(
    title=dict(
        text=f'Monthly Soil Moisture Comparison — Odisha Watershed ({START_DATE[:4]})',
        font=dict(size=16, color='#212121'),
        x=0.5
    ),
    barmode='group',
    xaxis_title='Month',
    yaxis_title='Soil Moisture (m³/m³)',
    plot_bgcolor=COLORS['bg'],
    paper_bgcolor='white',
    legend=dict(orientation='h', y=1.08, x=0.5, xanchor='center'),
    height=480,
    hovermode='x unified',
    font=dict(family='Arial', size=12),
    margin=dict(t=80, b=60)
)
fig.update_xaxes(showgrid=False, tickangle=-30)
fig.update_yaxes(showgrid=True, gridcolor=COLORS['grid'])
fig.show()

# ─────────────────────────────────────────────
# 7.  MAP — zoomed to watershed
# ─────────────────────────────────────────────
Map = geemap.Map()
Map.add_basemap('HYBRID')
Map.centerObject(roi, 9)          # zoomed to project area
Map.addLayer(roi, {'color': 'yellow'}, 'Watershed Boundary')

# Latest monthly composites clipped to ROI
latest_surface  = monthly_l4_surface.sort('system:time_start', False).first().clip(roi)
latest_rootzone = monthly_l4_rootzone.sort('system:time_start', False).first().clip(roi)

palette_sm  = ['#d73027','#f46d43','#fdae61','#fee090',
               '#e0f3f8','#abd9e9','#74add1','#313695']
vis_surface = {'min': 0.05, 'max': 0.45, 'palette': palette_sm}
vis_rz      = {'min': 0.10, 'max': 0.45, 'palette': palette_sm}

Map.addLayer(latest_surface,  vis_surface, 'L4 Surface SM (0–5 cm)')
Map.addLayer(latest_rootzone, vis_rz,      'L4 Root-Zone SM (0–100 cm)')

Map.add_colorbar(
    vis_params=vis_surface,
    label='Soil Moisture (m³/m³)',
    layer_name='L4 Surface SM (0–5 cm)',
    orientation='vertical'
)

print("🗺️  Map ready — toggle layers in the panel to switch between Surface and Root-Zone.")
Map

✅ Earth Engine initialised
Loading SMAP L4 (SPL4SMGP v007)…
   L4 images found: 2920
Loading SMAP L3 Enhanced (SPL3SMP_E v005)…
   L3 images found: 0

Extracting time series (this may take ~1-2 min)…
  ⚠️  No valid data for 'sm_l3_m3m3'
✅ Extracted 12 monthly records


🗺️  Map ready — toggle layers in the panel to switch between Surface and Root-Zone.


Map(center=[12.863556182712198, 77.87145656245913], controls=(WidgetControl(options=['position', 'transparent_…